# 2026 FIFA World Cup: Group Stage Dataset

Build a complete group stage match dataset with all 48 teams, 72 fixtures, dates, venues, climate classifications, and Elo ratings.


## 1. Imports

In [ ]:
import pandas as pd, numpy as np, re, requests
import matplotlib.pyplot as plt, seaborn as sns
from bs4 import BeautifulSoup

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

climate = pd.read_csv('../data/country_climate.csv')
historical = pd.read_csv('../data/fifa_world_cup_matches_enriched.csv', parse_dates=['Date'], keep_default_na=False)
print(f'Climate lookup: {len(climate)} countries')
print(f'Historical matches: {len(historical)}')

## 2. Scrape Group Stage from Wikipedia

In [ ]:
HEADERS = {'User-Agent': 'Mozilla/5.0'}
groups_2026 = {}
all_fixtures = []

for grp in ['A','B','C','D','E','F','G','H','I','J','K','L']:
    url = f'https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_Group_{grp}'
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        if resp.status_code != 200:
            print(f'Group {grp}: HTTP {resp.status_code}')
            continue
    except Exception as e:
        print(f'Group {grp}: {e}')
        continue
    
    soup = BeautifulSoup(resp.text, 'html.parser')
    
    # Extract teams from the group table
    for table in soup.find_all('table', class_='wikitable'):
        rows = table.find_all('tr')
        if not rows:
            continue
        header = [c.get_text(strip=True).lower() for c in rows[0].find_all(['th', 'td'])]
        if 'team' in header and 'draw' not in header:
            group_teams = []
            for row in rows[1:]:
                cells = row.find_all(['th', 'td'])
                if len(cells) >= 2:
                    t = cells[1].get_text(strip=True)
                    t = re.sub(r'\(H\)|\[.*?\]', '', t).strip()
                    if t and len(t) > 2 and not t.startswith('June') and not t[0].isdigit():
                        group_teams.append(t)
            if len(group_teams) >= 4:
                groups_2026[grp] = group_teams[:4]
                break
    
    # Extract match details from footballbox divs
    for fb in soup.find_all('div', class_='footballbox'):
        date_str = ''
        bday = fb.find('span', class_='bday')
        if bday:
            date_str = bday.get_text(strip=True)
        if not date_str:
            fdate = fb.find('div', class_='fdate')
            if fdate:
                date_str = fdate.get_text(strip=True)
        m = re.search(r'(\d{4}-\d{2}-\d{2})', date_str) if date_str else None
        if m:
            date_str = m.group(1)
        
        home_th = fb.find('th', class_='fhome')
        away_th = fb.find('th', class_='faway')
        home = re.sub(r'\[.*?\]|\(H\)', '', home_th.get_text(strip=True)).strip() if home_th else ''
        away = re.sub(r'\[.*?\]', '', away_th.get_text(strip=True)).strip() if away_th else ''
        
        fright = fb.find('div', class_='fright')
        venue = fright.get_text(' ', strip=True) if fright else ''
        venue_clean = re.sub(r'Referee:.*', '', venue).strip()
        city = venue_clean.split(',')[0].strip() if ',' in venue_clean else venue_clean[:60]
        
        all_fixtures.append({
            'Group': grp, 'Date': date_str,
            'Home_Team': home, 'Away_Team': away,
            'Host_City': city, 'Venue': venue_clean,
        })

fixtures = pd.DataFrame(all_fixtures)
fixtures['Date'] = pd.to_datetime(fixtures['Date'])
fixtures = fixtures.sort_values(['Date', 'Group']).reset_index(drop=True)

all_teams = sorted(set(t for teams in groups_2026.values() for t in teams))
print(f'Groups scraped: {len(groups_2026)}/12')
print(f'Fixtures: {len(fixtures)}')
print(f'Teams: {len(all_teams)}')
print(f'Date range: {fixtures["Date"].min().strftime("%Y-%m-%d")} to {fixtures["Date"].max().strftime("%Y-%m-%d")}')

## 3. Group Overview

In [ ]:
for grp in sorted(groups_2026.keys()):
    t = groups_2026[grp]
    print(f'Group {grp}: {t[0]:<25} {t[1]:<25} {t[2]:<25} {t[3]:<25}')
print()

for grp in sorted(fixtures['Group'].unique()):
    grp_fix = fixtures[fixtures['Group'] == grp]
    dates = grp_fix['Date'].dt.strftime('%Y-%m-%d').unique()
    print(f'Group {grp}: {len(grp_fix)} matches, {", ".join(dates)}')

## 4. Merge Climate Classification & Elo Ratings

In [ ]:
# ---- Merge Climate & Current Elo ----
team_climate = climate[climate['Country'].isin(all_teams)].set_index('Country')
missing = set(all_teams) - set(team_climate.index)
if missing: print(f'WARNING missing climate: {missing}')

# Use current Elo ratings computed from 49K international matches (through June 2026)
elo_current = pd.read_csv('../data/elo_current_2026.csv').set_index('Team')['Elo_Current']
team_elo = {}
for team in all_teams:
    if team in elo_current.index:
        team_elo[team] = float(elo_current[team])
    else:
        h = historical[historical['Home_Team'] == team]['Home_Elo'].tail(1)
        team_elo[team] = float(h.values[0]) if len(h) > 0 else float(historical['Home_Elo'].median())

for label, col in [('Home','Home_Team'), ('Away','Away_Team')]:
    fixtures[f'{label}_Koppen'] = fixtures[col].map(lambda t: team_climate.loc[t, 'Koppen'] if t in team_climate.index else '')
    fixtures[f'{label}_Climate_Zone'] = fixtures[col].map(lambda t: team_climate.loc[t, 'Climate_Zone'] if t in team_climate.index else '')
    fixtures[f'{label}_Warm_Climate'] = fixtures[col].map(lambda t: int(team_climate.loc[t, 'Warm_Climate']) if t in team_climate.index else 0)
    fixtures[f'{label}_Elo'] = fixtures[col].map(team_elo).round(1)

print('Merged climate and current Elo ratings')
print(f'Top 5 by Elo:')
for t, e in sorted(team_elo.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f'  {t:<25} {e:.0f}')

## 5. Tournament Temperature Classification

In [ ]:
# June average temperatures for 2026 host cities
city_temps = {
    'Estadio Azteca': 17.6, 'Estadio Akron': 22.5, 'Mercedes-Benz Stadium': 25.4, 'Estadio BBVA': 28.8,
    'BMO Field': 19.4, "Levi's Stadium": 20.1, 'SoFi Stadium': 19.8, 'BC Place': 15.2,
    'Lumen Field': 17.1, 'Hard Rock Stadium': 28.0, 'NRG Stadium': 28.3, 'AT&T Stadium': 28.5,
    'Arrowhead Stadium': 24.2, 'Lincoln Financial Field': 23.0, 'Gillette Stadium': 20.3, 'MetLife Stadium': 22.5,
}
for stadium, temp in city_temps.items():
    fixtures.loc[fixtures['Venue'].str.contains(stadium[:15], na=False), 'City_Temp'] = temp

fixtures['City_Temp'] = fixtures['City_Temp'].fillna(fixtures['City_Temp'].mean())
tournament_avg = fixtures['City_Temp'].mean()
fixtures['Tournament_Avg_Temp_C'] = round(tournament_avg, 1)
fixtures['Warm_Cup'] = 1 if tournament_avg > 19.3 else 0

print(f'Tournament average temperature: {tournament_avg:.1f} C')
print(f'Historical median: 19.3 C')
print(f'Classification: {"WARM CUP" if tournament_avg > 19.3 else "COOL CUP"}')

city_df = pd.DataFrame({'City': list(city_temps.keys()), 'Temp': list(city_temps.values())}).sort_values('Temp')
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#DA7B42' if t > 19.3 else '#2A7F8C' for t in city_df['Temp']]
ax.barh(city_df['City'], city_df['Temp'], color=colors, edgecolor='white')
ax.axvline(x=19.3, color='gray', linestyle='--', label='Historical median (19.3 C)')
ax.axvline(x=tournament_avg, color='#DA7B42', linewidth=2, label=f'2026 avg ({tournament_avg:.1f} C)')
ax.set_xlabel('June Average Temperature (C)')
ax.set_title('2026 Host City Temperatures')
ax.legend()
plt.tight_layout()

## 6. Team Overview

In [ ]:
team_df = pd.DataFrame({'Team': all_teams})
team_df['Koppen'] = team_df['Team'].map(lambda t: team_climate.loc[t, 'Koppen'] if t in team_climate.index else '')
team_df['Climate_Zone'] = team_df['Team'].map(lambda t: team_climate.loc[t, 'Climate_Zone'] if t in team_climate.index else '')
team_df['Warm_Climate'] = team_df['Team'].map(lambda t: int(team_climate.loc[t, 'Warm_Climate']) if t in team_climate.index else 0)
team_df['Elo'] = team_df['Team'].map(team_elo).round(0)
team_df = team_df.sort_values('Elo', ascending=False).reset_index(drop=True)

warm_count = int(team_df['Warm_Climate'].sum())
cool_count = len(team_df) - warm_count
print(f'Warm-climate teams: {warm_count} ({warm_count/len(team_df)*100:.0f}%)')
print(f'Cool-climate teams: {cool_count} ({cool_count/len(team_df)*100:.0f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
warm_t = team_df[team_df['Warm_Climate'] == 1].sort_values('Elo')
cool_t = team_df[team_df['Warm_Climate'] == 0].sort_values('Elo')
axes[0].barh(warm_t['Team'], warm_t['Elo'], color='#DA7B42', edgecolor='white')
axes[0].set_title(f'Warm-Climate Teams ({len(warm_t)})')
axes[1].barh(cool_t['Team'], cool_t['Elo'], color='#2A7F8C', edgecolor='white')
axes[1].set_title(f'Cool-Climate Teams ({len(cool_t)})')
for ax in axes:
    ax.set_xlabel('Elo Rating')
plt.tight_layout()
team_df.head(16)

## 7. Matchup Composition

In [ ]:
fixtures['Matchup'] = 'Warm vs Cool'
fixtures.loc[(fixtures['Home_Warm_Climate'] == 1) & (fixtures['Away_Warm_Climate'] == 1), 'Matchup'] = 'Warm vs Warm'
fixtures.loc[(fixtures['Home_Warm_Climate'] == 0) & (fixtures['Away_Warm_Climate'] == 0), 'Matchup'] = 'Cool vs Cool'

counts = fixtures['Matchup'].value_counts()
fig, ax = plt.subplots(figsize=(8, 5))
counts.plot(kind='bar', ax=ax, color=['#DA7B42', '#2A7F8C', 'mediumseagreen'], edgecolor='white')
ax.set_title('2026 Group Stage Matchup Types')
ax.set_xlabel('')
ax.set_ylabel('Matches')
for c in ax.containers:
    ax.bar_label(c, fontsize=12)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()

for t, c in counts.items():
    print(f'  {t}: {c} matches ({c/len(fixtures)*100:.0f}%)')

## 8. Save Dataset

In [ ]:
columns = ['Group', 'Date', 'Home_Team', 'Away_Team', 'Host_City',
            'Home_Koppen', 'Home_Climate_Zone', 'Home_Warm_Climate', 'Home_Elo',
            'Away_Koppen', 'Away_Climate_Zone', 'Away_Warm_Climate', 'Away_Elo',
            'City_Temp', 'Tournament_Avg_Temp_C', 'Warm_Cup']

out = fixtures[columns].copy()
out['Year'] = 2026
out['Stage'] = 'Group Stage'
out['Host_Country'] = 'USA, Canada, Mexico'

out.to_csv('../data/2026_group_stage.csv', index=False)
print(f'Saved ../data/2026_group_stage.csv ({len(out)} matches, {len(out.columns)} columns)')
print()
print('Dataset summary:')
print(f'  Tournament: {"WARM CUP" if tournament_avg > 19.3 else "COOL CUP"} ({tournament_avg:.1f} C)')
print(f'  Teams: {len(all_teams)} total, {warm_count} warm-climate, {cool_count} cool-climate')
print(f'  Groups: {len(groups_2026)} (A-L)')
print(f'  Fixtures: {len(fixtures)} group stage matches')
print(f'  Date range: {fixtures["Date"].min().strftime("%Y-%m-%d")} to {fixtures["Date"].max().strftime("%Y-%m-%d")}')
print()
print('To use: pd.read_csv("../data/2026_group_stage.csv", parse_dates=["Date"])')
out.head()

---

## 9. Base Elo Win Probabilities

Standard logistic Elo formula with home advantage for designated host nations only (USA, Mexico, Canada).


In [ ]:
HOST_NATIONS = {'United States', 'Mexico', 'Canada'}
ELO_HFA = 100  # home field advantage in Elo points

def elo_prob(home_elo, away_elo, home_is_host=False):
    """Standard Elo win probability for the home team.
    
    Args:
        home_elo: home team's Elo rating
        away_elo: away team's Elo rating  
        home_is_host: if True, apply +100 Elo boost (host nation only)
    """
    hfa = ELO_HFA if home_is_host else 0
    return 1.0 / (1.0 + 10.0 ** ((away_elo - (home_elo + hfa)) / 400.0))

# Apply to 2026 dataset
df = pd.read_csv('../data/2026_group_stage.csv', parse_dates=['Date'])

df['Home_Is_Host'] = df['Home_Team'].isin(HOST_NATIONS)
df['Away_Is_Host'] = df['Away_Team'].isin(HOST_NATIONS)

df['Home_Win_Prob'] = df.apply(
    lambda r: round(elo_prob(r['Home_Elo'], r['Away_Elo'], r['Home_Is_Host']), 4), axis=1
)
df['Draw_Prob'] = 0.0  # placeholder
df['Away_Win_Prob'] = round(1.0 - df['Home_Win_Prob'], 4)

print(f'Matches with HFA applied: {df["Home_Is_Host"].sum()}')
print(f'Host nations playing at home: {sorted(df[df["Home_Is_Host"]]["Home_Team"].unique())}')
print()

# Show matches where HFA is applied
host_matches = df[df['Home_Is_Host']]
print('Host nation home matches (with +100 Elo HFA):')
print(host_matches[['Group','Date','Home_Team','Away_Team','Home_Elo','Away_Elo','Home_Win_Prob']].to_string(index=False))

print(f'\nNon-host matches: {len(df) - len(host_matches)} (no HFA applied)')

In [ ]:
# Verify: host HFA should increase win probability vs no-HFA
print('HFA impact on host matches:')
for _, r in host_matches.iterrows():
    prob_without = r['Home_Elo_Expected'] if 'Home_Elo_Expected' in df.columns else elo_prob(r['Home_Elo'], r['Away_Elo'], False)
    prob_with = r['Home_Win_Prob']
    boost = prob_with - prob_without
    print(f'  {r["Home_Team"]} vs {r["Away_Team"]}: {prob_without:.3f} -> {prob_with:.3f} (+{boost:.3f} from HFA)')

In [ ]:
# Distribution of win probabilities
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(df['Home_Win_Prob'], bins=20, color='#2A7F8C', edgecolor='white', alpha=0.7, label='All matches')
if len(host_matches) > 0:
    ax.hist(host_matches['Home_Win_Prob'], bins=20, color='#DA7B42', edgecolor='white', alpha=0.7, label='Host home matches')
ax.set_xlabel('Home Win Probability')
ax.set_ylabel('Matches')
ax.set_title('Home Win Probability Distribution')
ax.legend()
plt.tight_layout()

print(f'Mean home win prob (all):  {df["Home_Win_Prob"].mean():.3f}')
print(f'Mean home win prob (host): {host_matches["Home_Win_Prob"].mean():.3f}')

In [ ]:
# Save updated dataset
prob_cols = ['Group','Date','Home_Team','Away_Team','Host_City',
             'Home_Koppen','Home_Climate_Zone','Home_Warm_Climate','Home_Elo',
             'Away_Koppen','Away_Climate_Zone','Away_Warm_Climate','Away_Elo',
             'Home_Is_Host','Away_Is_Host',
             'Home_Win_Prob','Away_Win_Prob','Draw_Prob',
             'City_Temp','Tournament_Avg_Temp_C','Warm_Cup']

df_out = df[prob_cols].copy()
df_out['Year'] = 2026
df_out['Stage'] = 'Group Stage'
df_out['Host_Country'] = 'USA, Canada, Mexico'

df_out.to_csv('../data/2026_group_stage.csv', index=False)
print(f'Saved ../data/2026_group_stage.csv ({len(df_out)} matches, {len(df_out.columns)} columns)')
print('Added: Home_Is_Host, Away_Is_Host, Home_Win_Prob, Away_Win_Prob, Draw_Prob')
df_out.head()

---

## 10. Monte Carlo Simulation (100,000 iterations)

Simulate the full tournament: 72 group stage matches → knockout bracket → Final. Update Elo dynamically with K=12 after each match.


In [53]:
# ---- Calibration from historical data ----
HOME_GOAL_BASE = 1.77  # avg home goals historically
AWAY_GOAL_BASE = 1.05  # avg away goals historically
ELO_AVG = df['Home_Elo'].mean()  # average Elo in 2026 field
K_FACTOR = 12  # simplified K for within-tournament Elo updates

In [54]:
# ---- Match simulation ----
def simulate_match(home_elo, away_elo, home_is_host=False):
    """Simulate a single match using Poisson goal model.
    Returns (home_goals, away_goals)."""
    hfa = 100 if home_is_host else 0
    # Expected goals using Elo-based scaling
    home_lambda = HOME_GOAL_BASE * np.exp((home_elo - ELO_AVG) / 400)
    away_lambda = AWAY_GOAL_BASE * np.exp((away_elo - ELO_AVG) / 400)
    # Apply HFA to goal expectations
    home_lambda *= 1.0 + (hfa / 2000)
    home_goals = np.random.poisson(max(0.1, home_lambda))
    away_goals = np.random.poisson(max(0.1, away_lambda))
    return home_goals, away_goals

def update_elo(home_elo, away_elo, home_goals, away_goals, home_is_host=False):
    """Update Elo ratings after a match. Returns (new_home_elo, new_away_elo)."""
    hfa = 100 if home_is_host else 0
    expected = 1.0 / (1.0 + 10.0 ** ((away_elo - (home_elo + hfa)) / 400.0))
    if home_goals > away_goals:
        result = 1.0
    elif home_goals < away_goals:
        result = 0.0
    else:
        result = 0.5
    new_home = home_elo + K_FACTOR * (result - expected)
    new_away = away_elo + K_FACTOR * (1.0 - result - (1.0 - expected))
    return new_home, new_away

---

## 10. Monte Carlo Simulation (100,000 iterations)

Simulate the full tournament: 72 group matches → 32-team knockout → Final. Elo updated dynamically with K=12 after each match. Poisson goal model with Elo-based expected goals.


In [ ]:
# ---- Simulation parameters ----
HOME_GOAL_BASE = 1.77  # historical avg home goals per match
AWAY_GOAL_BASE = 1.05
ELO_AVG = df['Home_Elo'].mean()
K_FACTOR = 12
HOST_NATIONS = {'United States', 'Mexico', 'Canada'}

# Pre-compute team data
all_teams = sorted(set(df['Home_Team'].unique()) | set(df['Away_Team'].unique()))
team_idx = {t: i for i, t in enumerate(all_teams)}
base_elo = np.array([df[df['Home_Team'] == t]['Home_Elo'].iloc[0] if t in df['Home_Team'].values
                      else df[df['Away_Team'] == t]['Away_Elo'].iloc[0] for t in all_teams])
is_host = np.array([t in HOST_NATIONS for t in all_teams])

teams_by_group = {}
for _, row in fixtures.iterrows():
    grp = row['Group']
    teams_by_group.setdefault(grp, set()).add(row['Home_Team'])
    teams_by_group[grp].add(row['Away_Team'])

print(f'Teams: {len(all_teams)}, Groups: {len(teams_by_group)}, Elo range: {base_elo.min():.0f}-{base_elo.max():.0f}')

In [56]:
# ---- Core functions ----
def match_result(he, ae, hfa=0):
    """Simulate match with Poisson model. Returns (home_goals, away_goals)."""
    hl = HOME_GOAL_BASE * np.exp((he - ELO_AVG) / 400) * (1.0 + hfa / 2000)
    al = AWAY_GOAL_BASE * np.exp((ae - ELO_AVG) / 400)
    return np.random.poisson(max(0.1, hl)), np.random.poisson(max(0.1, al))

def elo_update(he, ae, hg, ag, hfa=0):
    """Update Elo ratings. Returns (new_he, new_ae)."""
    exp_h = 1.0 / (1.0 + 10.0 ** ((ae - (he + hfa)) / 400.0))
    if hg > ag: rh = 1.0
    elif hg < ag: rh = 0.0
    else: rh = 0.5
    return he + K_FACTOR * (rh - exp_h), ae + K_FACTOR * (1.0 - rh - (1.0 - exp_h))

def knockout(te1, te2, e1, e2):
    """Simulate knockout match, return (winner, new_e1, new_e2)."""
    hg, ag = match_result(e1, e2)
    ne1, ne2 = elo_update(e1, e2, hg, ag)
    if hg > ag: return te1, ne1, ne2
    elif hg < ag: return te2, ne2, ne1
    else:  # penalty coin flip
        if np.random.random() < 0.5: return te1, ne1, ne2
        else: return te2, ne2, ne1

In [ ]:
# ---- Monte Carlo with Smart K-factor ----
N_SIMS = 100_000
np.random.seed(42)

# K-factor: scales by tournament stage and opponent parity
# Group x0.5, R32 x0.75, R16 x1.0, QF x1.25, SF x1.5, Final x2.0
# Parity: closely matched teams get higher K (more informative)
# Upset bonus: underdog wins get 1.5x multiplier
def smart_K(he, ae, stage_wt=1.0, hfa=0):
    K_BASE = 12
    exp_h = 1.0 / (1.0 + 10.0 ** ((ae - (he + hfa)) / 400.0))
    parity = max(0.2, 1.0 - abs(exp_h - 0.5) * 1.6)  # 0.2-1.0
    return K_BASE * stage_wt * parity

def elo_update(he, ae, hg, ag, stage_wt=1.0, hfa=0):
    exp_h = 1.0 / (1.0 + 10.0 ** ((ae - (he + hfa)) / 400.0))
    if hg > ag: rh = 1.0
    elif hg < ag: rh = 0.0
    else: rh = 0.5
    K = smart_K(he, ae, stage_wt, hfa)
    if abs(rh - exp_h) > 0.4: K *= 1.5  # upset bonus
    return he + K * (rh - exp_h), ae + K * ((1.0 - rh) - (1.0 - exp_h))

def knockout(te1, te2, e1, e2, sw=1.0):
    hg, ag = match_result(e1, e2)
    ne1, ne2 = elo_update(e1, e2, hg, ag, sw)
    if hg > ag: return te1, ne1, ne2
    elif hg < ag: return te2, ne2, ne1
    else:
        if np.random.random() < 0.5: return te1, ne1, ne2
        else: return te2, ne2, ne1

stages = ['Group', 'R32', 'R16', 'QF', 'SF', 'Final', 'Winner']
results = {t: {s: 0 for s in stages} for t in all_teams}

print(f'Running {N_SIMS:,} full tournament simulations with smart K...')

for sim in range(N_SIMS):
    elos = base_elo.copy()
    
    # ---- Group Stage (K x0.5) ----
    standings = {grp: {t: {'Pts': 0, 'GF': 0, 'GA': 0, 'GD': 0} for t in teams} for grp, teams in teams_by_group.items()}
    for _, row in fixtures.iterrows():
        grp, h, a = row['Group'], row['Home_Team'], row['Away_Team']
        hi, ai = team_idx[h], team_idx[a]
        hfa = 100 if is_host[hi] else 0
        hg, ag = match_result(elos[hi], elos[ai], hfa)
        elos[hi], elos[ai] = elo_update(elos[hi], elos[ai], hg, ag, 0.5, hfa)
        standings[grp][h]['GF'] += hg; standings[grp][h]['GA'] += ag
        standings[grp][a]['GF'] += ag; standings[grp][a]['GA'] += hg
        standings[grp][h]['GD'] = standings[grp][h]['GF'] - standings[grp][h]['GA']
        standings[grp][a]['GD'] = standings[grp][a]['GF'] - standings[grp][a]['GA']
        if hg > ag: standings[grp][h]['Pts'] += 3
        elif ag > hg: standings[grp][a]['Pts'] += 3
        else: standings[grp][h]['Pts'] += 1; standings[grp][a]['Pts'] += 1
    
    ranked = {grp: sorted(d.items(), key=lambda x: (x[1]['Pts'], x[1]['GD'], x[1]['GF']), reverse=True) for grp, d in standings.items()}
    thirds = [(grp, order[2][0], order[2][1]) for grp, order in ranked.items()]
    thirds.sort(key=lambda x: (x[2]['Pts'], x[2]['GD'], x[2]['GF']), reverse=True)
    
    for _, order in ranked.items():
        for team, _ in order: results[team]['Group'] += 1
    
    pool = []
    for _, order in ranked.items():
        pool.append((order[0][0], elos[team_idx[order[0][0]]]))
        pool.append((order[1][0], elos[team_idx[order[1][0]]]))
    for _, team, _ in thirds[:8]:
        pool.append((team, elos[team_idx[team]]))
    np.random.shuffle(pool)
    
    # ---- Knockout with escalating K ----
    for rnd, target, sw in [('R32', 32, 0.75), ('R16', 16, 1.0), ('QF', 8, 1.25)]:
        nxt = []
        for j in range(0, target, 2):
            t1, e1 = pool[j]; t2, e2 = pool[j+1]
            w, ne1, ne2 = knockout(t1, t2, e1, e2, sw)
            elos[team_idx[t1]] = ne1; elos[team_idx[t2]] = ne2
            nxt.append((w, elos[team_idx[w]]))
            results[t1][rnd] += 1; results[t2][rnd] += 1
        pool = nxt
    
    # ---- SF (K x1.5) ----
    sf_w, sf_l = [], []
    for j in range(0, 4, 2):
        t1, e1 = pool[j]; t2, e2 = pool[j+1]
        w, ne1, ne2 = knockout(t1, t2, e1, e2, 1.5)
        l = t2 if w == t1 else t1
        elos[team_idx[t1]] = ne1; elos[team_idx[t2]] = ne2
        sf_w.append((w, elos[team_idx[w]])); sf_l.append((l, elos[team_idx[l]]))
        results[t1]['SF'] += 1; results[t2]['SF'] += 1
    
    # ---- Final (K x2.0) + 3rd place (K x1.0) ----
    t1, e1 = sf_w[0]; t2, e2 = sf_w[1]
    champ, _, _ = knockout(t1, t2, e1, e2, 2.0)
    results[t1]['Final'] += 1; results[t2]['Final'] += 1
    results[champ]['Winner'] += 1
    
    t3, e3 = sf_l[0]; t4, e4 = sf_l[1]
    knockout(t3, t4, e3, e4, 1.0)
    results[t3]['Final'] += 1; results[t4]['Final'] += 1
    
    if (sim + 1) % 25000 == 0:
        print(f'  {sim+1:,}/{N_SIMS:,} ({(sim+1)/N_SIMS*100:.0f}%)')

In [ ]:
# ---- Aggregate ----
sim_df = pd.DataFrame(results).T
sim_df.index.name = 'Team'

for stage in ['Group', 'R32', 'R16', 'QF', 'SF', 'Final', 'Winner']:
    sim_df[f'{stage}_Prob'] = (sim_df[stage] / N_SIMS).round(4)

# Group probability: all 48 teams always participate in group stage = 100%
# R32: reaching knockout
sim_df['Advance_Prob'] = sim_df['R32_Prob']

sim_df = sim_df.sort_values('Winner_Prob', ascending=False)

print('TOP 20 TOURNAMENT WIN PROBABILITY')
cols = ['Winner_Prob','Final_Prob','SF_Prob','QF_Prob','R16_Prob','R32_Prob']
print(sim_df[cols].head(20).to_string())
print(f'\n{len(sim_df)} teams simulated, {N_SIMS:,} iterations')

In [ ]:
# ---- Visualisation ----
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Win probability
top25 = sim_df.head(25).sort_values('Winner_Prob')
axes[0].barh(top25.index, top25['Winner_Prob'] * 100, color='#DA7B42', edgecolor='white')
for i, (_, r) in enumerate(top25.iterrows()):
    axes[0].text(r['Winner_Prob'] * 100 + 0.5, i, f"{r['Winner_Prob']*100:.2f}%", va='center', fontsize=9)
axes[0].set_xlabel('Win Probability (%)')
axes[0].set_title(f'2026 World Cup Win Probability (Top 25, {N_SIMS:,} sims)')

# Stage progression
top10 = sim_df.head(10)
stage_df = top10[['R32_Prob','R16_Prob','QF_Prob','SF_Prob','Final_Prob','Winner_Prob']].copy()
stage_df.columns = ['R32','R16','QF','SF','Final','Win']
stage_df.T.plot(kind='line', ax=axes[1], marker='o', linewidth=2)
axes[1].set_title('Stage Progression Probability (Top 10)')
axes[1].set_xlabel('Stage')
axes[1].set_ylabel('Probability')
axes[1].legend(bbox_to_anchor=(1.05, 1), fontsize=8)
axes[1].set_xticks(range(len(['R32','R16','QF','SF','Final','Win'])))
axes[1].set_xticklabels(['R32','R16','QF','SF','Final','Win'], rotation=0)

plt.tight_layout()

In [ ]:
# ---- Save ----
sim_df.to_csv('../data/2026_monte_carlo_results.csv')
print(f'Saved ../data/2026_monte_carlo_results.csv ({len(sim_df)} teams, {N_SIMS:,} iterations)')